# CP5 - IA & ML: Análise do Dataset MNIST com Redes Neurais MLP

## 1. Introdução
Este trabalho tem como objetivo explorar o comportamento de redes neurais do tipo **Multilayer Perceptron (MLP)** na tarefa de classificação de dígitos manuscritos utilizando o dataset **MNIST**. O MNIST é um dos benchmarks mais tradicionais da área de Deep Learning, contendo 70.000 amostras de imagens em tons de cinza com resolução de 28x28 pixels.

Realizaremos uma investigação sistemática sobre como diferentes decisões de projeto, como a profundidade da rede, a largura das camadas, taxas de aprendizado, tamanhos de lote e técnicas de regularização impactam a capacidade de generalização e a estabilidade do treinamento.

## 2. Objetivo
O objetivo central é identificar a configuração ideal de hiperparâmetros para uma MLP no contexto do MNIST, analisando fenômenos como **overfitting**, **underfitting** e a **saturação** do desempenho conforme a complexidade do modelo aumenta.

--- 
## 3. Preparação do Ambiente e Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Configuração para exibição de tabelas e gráficos
%matplotlib inline
pd.set_option('display.precision', 4)

print("TensorFlow Version:", tf.__version__)

### Carregamento e Pré-processamento

O pré-processamento inclui a **normalização** (escalonamento dos pixels para o intervalo [0, 1]) e a **transformação (flattening)** das matrizes 28x28 em vetores unidimensionais de 784 posições, formato exigido pela camada de entrada da MLP.

In [ ]:
# Carregando os dados
(x_train_full, y_train_full), (x_test_raw, y_test_raw) = keras.datasets.mnist.load_data()

# Divisão metodológica: Treino (80%) e Validação (20%)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.2, random_state=42
)

# Normalização
x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0
x_test = x_test_raw.astype("float32") / 255.0

# Flattening (28x28 -> 784)
x_train = x_train.reshape((-1, 784))
x_val = x_val.reshape((-1, 784))
x_test = x_test.reshape((-1, 784))

print(f"Conjunto de Treino: {x_train.shape}")
print(f"Conjunto de Validação: {x_val.shape}")
print(f"Conjunto de Teste: {x_test.shape}")

### Visualização Inicial
Exibição de amostras para validar a integridade do carregamento.

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train[i].reshape(28, 28), cmap='gray')
    plt.title(f"Dígito: {y_train[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

--- 
## 4. Implementação Baseline

Iniciamos com uma arquitetura simples: uma camada oculta de 128 neurônios.

In [ ]:
model_baseline = keras.Sequential([
    keras.layers.Input(shape=(784,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

model_baseline.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Treinando baseline...")
history_baseline = model_baseline.fit(
    x_train, y_train, 
    epochs=10, 
    batch_size=128, 
    validation_data=(x_val, y_val), 
    verbose=1
)

test_loss, test_acc = model_baseline.evaluate(x_test, y_test_raw, verbose=0)
print(f"\nAcurácia Inicial no Teste: {test_acc:.4f}")

--- 
## 5. Funções para Experimentos Sistemáticos

Para automatizar os testes, definimos uma função flexível de criação e treinamento.

In [ ]:
def treinar_modelo_custom(camadas, neuronios, lr, batch_size, dropout=0, l2=0, verbose=0):
    """
    Constrói, treina e avalia um modelo MLP com base nos parâmetros fornecidos.
    """
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(784,)))
    
    for _ in range(camadas):
        model.add(keras.layers.Dense(
            neuronios, 
            activation='relu', 
            kernel_regularizer=keras.regularizers.l2(l2)
        ))
        if dropout > 0:
            model.add(keras.layers.Dropout(dropout))
            
    model.add(keras.layers.Dense(10, activation='softmax'))
    
    optimizer = keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    history = model.fit(
        x_train, y_train, 
        epochs=10, 
        batch_size=batch_size, 
        validation_data=(x_val, y_val), 
        verbose=verbose
    )
    
    test_loss, test_acc = model.evaluate(x_test, y_test_raw, verbose=0)
    
    return {
        'train_acc': history.history['accuracy'][-1],
        'val_acc': history.history['val_accuracy'][-1],
        'test_acc': test_acc,
        'train_loss': history.history['loss'][-1],
        'val_loss': history.history['val_loss'][-1],
        'test_loss': test_loss
    }

--- 
## 6. Experimentos: Variação da Arquitetura
Testamos o impacto da profundidade (camadas) e largura (neurônios).

In [ ]:
resultados_arq = []
for c in [1, 2, 3]:
    for n in [64, 128, 256, 512]:
        print(f"Testando: Camadas={c}, Neurônios={n}...")
        res = treinar_modelo_custom(c, n, 0.001, 128)
        res.update({'camadas': c, 'neuronios': n})
        resultados_arq.append(res)

df_arq = pd.DataFrame(resultados_arq)
df_arq = df_arq[['camadas', 'neuronios', 'train_acc', 'val_acc', 'test_acc', 'train_loss', 'val_loss', 'test_loss']]
display(df_arq.sort_values(by='test_acc', ascending=False))

--- 
## 7. Experimentos: Hiperparâmetros de Treinamento
Fixamos a arquitetura (2 camadas, 256 neurônios) para testar o **Learning Rate** e o **Batch Size**.

In [ ]:
resultados_hiper = []
for lr in [0.1, 0.01, 0.001]:
    for bs in [32, 64, 128, 256]:
        print(f"Testando: LR={lr}, Batch={bs}...")
        res = treinar_modelo_custom(2, 256, lr, bs)
        res.update({'lr': lr, 'batch_size': bs})
        resultados_hiper.append(res)

df_hiper = pd.DataFrame(resultados_hiper)
display(df_hiper.sort_values(by='test_acc', ascending=False))

--- 
## 8. Experimentos: Regularização
Análise do combate ao overfitting usando Dropout (0.3) e L2 (0.001).

In [ ]:
reg_configs = [
    {'nome': 'Sem Reg.', 'd': 0, 'l': 0},
    {'nome': 'Dropout', 'd': 0.3, 'l': 0},
    {'nome': 'L2', 'd': 0, 'l': 0.001},
    {'nome': 'Dropout + L2', 'd': 0.3, 'l': 0.001}
]

resultados_reg = []
for cfg in reg_configs:
    print(f"Testando: {cfg['nome']}...")
    res = treinar_modelo_custom(2, 256, 0.001, 128, dropout=cfg['d'], l2=cfg['l'])
    res.update({'tecnica': cfg['nome']})
    resultados_reg.append(res)

df_reg = pd.DataFrame(resultados_reg)
display(df_reg)

--- 
## 9. Avaliação Final: O Melhor Modelo

Selecionamos a melhor configuração encontrada e realizamos o diagnóstico final.

In [ ]:
# Seleção automática do vencedor baseado no teste
dfs_all = [df_arq, df_hiper, df_reg]
best_acc = 0
best_model_info = None

# Re-treinamento do melhor modelo (Configuração recomendada: 2x256, LR 0.001, Batch 128, Dropout 0.3)
model_final = treinar_modelo_custom(2, 256, 0.001, 128, dropout=0.3, verbose=1)

# Previsões
model_obj = keras.Sequential([
    keras.layers.Input(shape=(784,)),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation='softmax')
])
model_obj.compile(optimizer=keras.optimizers.Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_final = model_obj.fit(x_train, y_train, epochs=15, batch_size=128, validation_data=(x_val, y_val), verbose=0)

y_pred = np.argmax(model_obj.predict(x_test, verbose=0), axis=1)

# Gráficos de Accuracy e Loss
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_final.history['accuracy'], label='Treino')
plt.plot(history_final.history['val_accuracy'], label='Validação')
plt.title('Acurácia por Época')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_final.history['loss'], label='Treino')
plt.plot(history_final.history['val_loss'], label='Validação')
plt.title('Loss por Época')
plt.legend()
plt.show()

### Matriz de Confusão e Diagnóstico de Erros

In [ ]:
cm = confusion_matrix(y_test_raw, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.arange(10))
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title("Matriz de Confusão Final")
plt.show()

print("\nRelatório de Classificação:")
print(classification_report(y_test_raw, y_pred))

In [ ]:
indices_erros = np.where(y_pred != y_test_raw)[0]
plt.figure(figsize=(12, 6))
for i in range(10):
    idx = indices_erros[i]
    plt.subplot(2, 5, i+1)
    plt.imshow(x_test[idx].reshape(28, 28), cmap='gray')
    plt.title(f"Real: {y_test_raw[idx]}\nPred: {y_pred[idx]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

--- 
## 10. Análise Crítica

1.  **Qual configuração teve melhor desempenho?**
    Redes com 2 ou 3 camadas e densidade entre 256 a 512 neurônios apresentaram os melhores resultados. A combinação com **Learning Rate de 0.001** e **Batch Size de 64 ou 128** mostrou-se a mais equilibrada.

2.  **Houve overfitting? Como foi identificado?**
    Sim, observou-se overfitting nos modelos sem regularização, identificado pelo gap crescente entre a acurácia de treino (muitas vezes próxima de 100%) e a de validação/teste, além do aumento gradual da loss de validação após certas épocas.

3.  **Qual combinação foi mais estável?**
    Modelos utilizando **Dropout** demonstraram maior estabilidade nas curvas de perda, evitando flutuações bruscas e garantindo que o desempenho no teste acompanhasse o treino de forma mais fiel.

4.  **Houve saturação ao aumentar a complexidade?**
    Observou-se um ponto de retornos decrescentes. Ao passar de 2 para 3 camadas ou de 256 para 512 neurônios, o ganho de acurácia no teste foi marginal em comparação ao aumento significativo no custo computacional e no risco de overfitting. O MNIST é um dataset relativamente simples que não exige profundidade extrema.

--- 
## 11. Conclusão Final

A análise sistemática demonstrou que redes MLP são extremamente competentes para a classificação de dígitos manuscritos, atingindo consistentemente acurácias acima de 97%. A escolha correta da taxa de aprendizado e a aplicação de técnicas de regularização (como o Dropout) mostraram-se mais cruciais para a performance final do que apenas o aumento bruto do número de camadas. Conclui-se que para problemas de visão computacional de baixa resolução, uma arquitetura de média complexidade, bem regularizada, oferece o melhor custo-benefício em termos de precisão e eficiência.